In [20]:
import torch
import torch.nn as nn
import torch.optim as optim

In [22]:
corpus = "we are learning cbow model".split()
vocab = sorted(set(corpus))

word_to_ix = {w:i for i,w in enumerate(vocab)}
ix_to_word = {i:w for w,i in word_to_ix.items()}

vocab_size = len(vocab)
window = 1

In [23]:
data = []
for i, word in enumerate(corpus):
    context = []
    for j in [i-1, i+1]:
        if 0 <= j < len(corpus):
            context.append(corpus[j])
    if len(context) > 0:
        data.append((context, word))

data

[(['are'], 'we'),
 (['we', 'learning'], 'are'),
 (['are', 'cbow'], 'learning'),
 (['learning', 'model'], 'cbow'),
 (['cbow'], 'model')]

In [24]:
def one_hot(word):
    v = torch.zeros(vocab_size)
    v[word_to_ix[word]] = 1
    return v

X = []
Y = []

for ctx, tgt in data:
    ctx_vec = sum(one_hot(w) for w in ctx) / len(ctx)
    X.append(ctx_vec)
    Y.append(torch.tensor([word_to_ix[tgt]]))

X = torch.stack(X)
Y = torch.stack(Y).squeeze()

In [25]:
class CBOW(nn.Module):
    def __init__(self, vocab, dim=8):
        super().__init__()
        self.emb = nn.Linear(vocab, dim)
        self.out = nn.Linear(dim, vocab)

    def forward(self, x):
        return self.out(self.emb(x))

model = CBOW(vocab_size)
loss_fn = nn.CrossEntropyLoss()
opt = optim.Adam(model.parameters(), lr=0.01)

In [26]:
for epoch in range(200):
    out = model(X)
    loss = loss_fn(out, Y)

    opt.zero_grad()
    loss.backward()
    opt.step()

    if epoch % 40 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 1.6455549001693726
Epoch: 40 Loss: 0.6856106519699097
Epoch: 80 Loss: 0.17518383264541626
Epoch: 120 Loss: 0.0548822358250618
Epoch: 160 Loss: 0.02452811785042286


In [27]:
def predict(ctx_words):
    v = sum(one_hot(w) for w in ctx_words) / len(ctx_words)
    out = model(v)
    return ix_to_word[torch.argmax(out).item()]

predict(["we", "learning"])

'are'